# 03 — GeoDataFrame QA

Build a GeoDataFrame from lat/lon, run geometry checks, reproject before metric ops, and join points to nearest neighbors with a real distance column.

In [ ]:
import geopandas as gpd

from geo_daily_tools.sample_data import messy_sensor_records
from geo_daily_tools.validation import validate_sensor_records
from geo_daily_tools.geo_validation import (
    points_from_latlon,
    geometry_quality_report,
    geometry_type_summary,
    bounds_summary,
    require_crs,
    reproject_if_needed,
    nearest_neighbor_join,
)

valid_df, _ = validate_sensor_records(messy_sensor_records())
gdf = points_from_latlon(valid_df)
gdf.head()

## Geometry QA

In [ ]:
geometry_quality_report(gdf)

In [ ]:
geometry_type_summary(gdf)

In [ ]:
bounds_summary(gdf)

## CRS hygiene

EPSG:4326 is fine for *storing* lat/lon. Always reproject to a projected CRS (e.g. EPSG:3857 or a local UTM zone) before computing distance or area.

In [ ]:
require_crs(gdf, expected_crs='EPSG:4326')
gdf_m = reproject_if_needed(gdf, 'EPSG:3857')
gdf_m.crs

## Nearest-neighbor join with a meaningful distance column

In [ ]:
target = gpd.GeoDataFrame(
    {'name': ['centroid']},
    geometry=gpd.points_from_xy([gdf_m.geometry.x.mean()], [gdf_m.geometry.y.mean()]),
    crs=gdf_m.crs,
)
joined = nearest_neighbor_join(gdf_m, target)
joined[['obs_id', 'name', 'distance']]